# Sprint 2 — Notebook final : Pipeline d'indexation hybride : Vecteurs + Graphe de connaissances 

**PFE :** Agentic GraphRAG | **Étudiante :** Wiame Anejjar | **Encadrant :** Pr. Abdelaaziz Hessane


Ce notebook est la **version finale** et contient :
- Logging JSON strict `{entities:[], relations:[]}` + **taux de parsing**
- Fix warnings LightRAG (`complete delimiter` manquant, relations 4/5 champs) via `sanitize_extraction_output()`
- Évaluation quantitative : **latence**, **RAGAS faithfulness**, **nodes traversed approx**
- Export Neo4j fiable depuis **GraphML** (import batch)
- **LLM local uniquement** (Ollama), **pas Groq**


## 0) Installation (une seule fois)

### 0.1 — Nettoyage des dépendances

Ce script prépare l'environnement en supprimant les packages conflictuels qui peuvent causer des erreurs d'importation lors de l'utilisation de RAGAS et LangChain. Cette étape garantit que les dépendances ultérieures s'installent correctement sans conflit de versioning.

In [20]:
import subprocess, sys

pkgs = [
    "ragas",
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-text-splitters",
    "langsmith",
    "openai"
]

for p in pkgs:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", p]
    )

print("Nettoyage terminé")

Nettoyage terminé


### 0.2 — Vérification de l'import RAGAS et LangChain

Validation que les packages critiques (RAGAS, LangChain, embeddings Ollama) sont correctement installés et importables. Cette vérification préalable évite les erreurs durant l'exécution des cellules ultérieures.

In [1]:
from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

from langchain_community.chat_models import ChatOllama

print("OK")

c:\Users\ADMIN\Desktop\PFE_Agentic_Graphrag\GraphRag_PFE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23352\3722685968.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23352\3722685968.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23352\3722685968.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23

## 1) Configuration (LOCAL Ollama)

### 1.1 — Configuration des chemins et variables d'environnement

Initialisation complète de l'environnement du projet : chemins des répertoires (data, indexes, output), paramètres Ollama (URL, modèle, dimension embedding), paramètres LightRAG (taille chunks, gleaning, tokens max), et configuration Neo4j. Cette centralisation des paramètres facilite les ajustements futurs.

In [10]:
import os, json, time, shutil, asyncio, requests
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any

import numpy as np
from dotenv import load_dotenv
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()
load_dotenv()

BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data' / 'processed'
INDEX_DIR = BASE_DIR / 'indexes' / 'lightrag_500_connected_v2'
OUT_DIR = DATA_DIR / 'triplets'
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = DATA_DIR / 'checkpoint_500_connected_v2.json'

OLLAMA_URL = os.getenv('OLLAMA_URL', 'http://localhost:11434').strip().rstrip('/')
MODEL_NAME = os.getenv('MODEL_NAME', 'llama3.1:8b').strip()
EMBED_MODEL = os.getenv('EMBED_MODEL', 'nomic-embed-text').strip()
EMBED_DIM = int(os.getenv('EMBED_DIM', '768'))

NEO4J_URI = os.getenv('NEO4J_URI', '').strip()
NEO4J_USER = os.getenv('NEO4J_USER', os.getenv('NEO4J_USERNAME','neo4j')).strip()
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '').strip()

NB_DOCS = int(os.getenv('NB_DOCS', '500'))

# Reglages stabilite + densite
CHUNK_TOKEN_SIZE = int(os.getenv('CHUNK_TOKEN_SIZE', '700'))
CHUNK_OVERLAP = int(os.getenv('CHUNK_OVERLAP', '120'))
MAX_EXTRACT_INPUT_TOKENS = int(os.getenv('MAX_EXTRACT_INPUT_TOKENS', '6000'))

ENTITY_EXTRACT_MAX_GLEANING = int(os.getenv('ENTITY_EXTRACT_MAX_GLEANING', '6'))
MAX_TOKENS_EXTRACT = int(os.getenv('MAX_TOKENS_EXTRACT', '1800'))
MAX_TOKENS_QUERY = int(os.getenv('MAX_TOKENS_QUERY', '900'))
OLLAMA_NUM_CTX = int(os.getenv('OLLAMA_NUM_CTX', '8192'))
LLM_MAX_ASYNC = int(os.getenv('LLM_MAX_ASYNC', '1'))
MAX_PARALLEL_INSERT = int(os.getenv('MAX_PARALLEL_INSERT', '1'))
SLEEP_BETWEEN_DOCS_S = float(os.getenv('SLEEP_BETWEEN_DOCS_S', '0.2'))

print('Config:')
print(' INDEX_DIR=', INDEX_DIR)
print(' NB_DOCS=', NB_DOCS)
print(' OLLAMA MODEL_NAME=', MODEL_NAME)
print(' EMBED_MODEL=', EMBED_MODEL, 'dim=', EMBED_DIM)
print(' gleaning=', ENTITY_EXTRACT_MAX_GLEANING)
print(' chunk_token_size=', CHUNK_TOKEN_SIZE, 'overlap=', CHUNK_OVERLAP)
print(' MAX_EXTRACT_INPUT_TOKENS=', MAX_EXTRACT_INPUT_TOKENS)
print(' MAX_TOKENS_EXTRACT=', MAX_TOKENS_EXTRACT)
print(' OLLAMA_NUM_CTX=', OLLAMA_NUM_CTX)


Config:
 INDEX_DIR= indexes\lightrag_500_connected_v2
 NB_DOCS= 500
 OLLAMA MODEL_NAME= llama3.1:8b
 EMBED_MODEL= nomic-embed-text dim= 768
 gleaning= 6
 chunk_token_size= 700 overlap= 120
 MAX_EXTRACT_INPUT_TOKENS= 6000
 MAX_TOKENS_EXTRACT= 1800
 OLLAMA_NUM_CTX= 8192


## 2) Logger JSON strict + taux de parsing (métrique qualité)

### 2.1 — Classe TripletLogger et système de parsing JSON strict

Implémentation d'un logger personnalisé qui enregistre les résultats de l'extraction LLM en format JSON structuré `{entities:[], relations:[]}`. Inclut le parsing strict avec métriques de qualité (taux de parsing, délimiteurs manquants, lignes malformées). Cette classe garantit la traçabilité complète de l'extraction et permet l'audit qualité ultérieur.

In [3]:
try:
    from lightrag.prompt import PROMPTS
    TUPLE_DELIM = PROMPTS.get('DEFAULT_TUPLE_DELIMITER', '<|#|>')
    COMPLETE_DELIM = PROMPTS.get('DEFAULT_COMPLETION_DELIMITER', '<|COMPLETE|>')

    # Graph quality rules: fewer duplicate nodes, more useful cross-document relations.
    PROMPTS['entity_extraction_system_prompt'] += """
---Additional Graph Quality Rules---
Extract reusable scientific entities, not full paper titles.
Do not extract the complete document title as an entity unless it is a named model, dataset, benchmark, tool, or method.

Use canonical names consistently:
- LLM, LLMs, Large Language Model, Large Language Models -> Large Language Models
- VLM, VLMs, Vision-Language Model(s) -> Vision-Language Models
- GNN, GNNs, Graph Neural Network(s) -> Graph Neural Networks
- RAG, Retrieval Augmented Generation -> Retrieval-Augmented Generation

Avoid isolated entities. Every extracted entity should participate in at least one relationship.
Prefer fewer high-quality entities over many weak entities.
For each chunk, extract as many meaningful relationships as possible between methods, models, datasets, tasks, metrics, benchmarks, applications, and concepts.
Use specific predicates such as uses, improves, evaluates_on, trained_on, applied_to, part_of, based_on, compares_with, extends, addresses.
Avoid generic predicates like related_to.
"""
except Exception:
    TUPLE_DELIM = '<|#|>'
    COMPLETE_DELIM = '<|COMPLETE|>'

@dataclass
class TripletLogger:
    total_llm_calls: int = 0
    parse_success: int = 0
    parse_failed: int = 0
    missing_complete_delimiter: int = 0
    malformed_entity_lines: int = 0
    malformed_relation_lines: int = 0
    total_entities: int = 0
    total_relations: int = 0
    items: list[dict[str, Any]] = field(default_factory=list)

    def parse_extraction(self, text: str, doc_id: str | None = None) -> dict[str, Any]:
        self.total_llm_calls += 1
        if COMPLETE_DELIM not in text:
            self.missing_complete_delimiter += 1

        entities, relations = [], []
        raw_lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        try:
            for ln in raw_lines:
                if ln == COMPLETE_DELIM:
                    break
                parts = ln.split(TUPLE_DELIM)
                if not parts:
                    continue

                if parts[0] == 'entity':
                    if len(parts) < 4:
                        self.malformed_entity_lines += 1
                        continue
                    entities.append({'name': parts[1].strip(), 'type': parts[2].strip(), 'description': parts[3].strip()})

                elif parts[0] == 'relation':
                    if len(parts) == 4:
                        self.malformed_relation_lines += 1
                        parts = parts + ['']
                    if len(parts) < 5:
                        self.malformed_relation_lines += 1
                        continue
                    relations.append({
                        'source': parts[1].strip(),
                        'target': parts[2].strip(),
                        'relation_keywords': parts[3].strip(),
                        'description': parts[4].strip(),
                    })

            self.parse_success += 1
            self.total_entities += len(entities)
            self.total_relations += len(relations)
            payload = {'doc_id': doc_id, 'entities': entities, 'relations': relations}
            self.items.append(payload)
            return payload
        except Exception:
            self.parse_failed += 1
            payload = {'doc_id': doc_id, 'entities': [], 'relations': [], 'raw': text[:400]}
            self.items.append(payload)
            return payload

    def save(self, path: Path):
        meta = {
            'total_llm_calls': self.total_llm_calls,
            'parse_success': self.parse_success,
            'parse_failed': self.parse_failed,
            'parse_success_pct': round(self.parse_success / max(1, self.total_llm_calls) * 100, 2),
            'missing_complete_delimiter': self.missing_complete_delimiter,
            'malformed_entity_lines': self.malformed_entity_lines,
            'malformed_relation_lines': self.malformed_relation_lines,
            'total_entities': self.total_entities,
            'total_relations': self.total_relations,
            'tuple_delimiter': TUPLE_DELIM,
            'completion_delimiter': COMPLETE_DELIM,
        }
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({'metadata': meta, 'triplets': self.items}, f, ensure_ascii=False, indent=2)
        print('Saved:', path)

triplet_logger = TripletLogger()
print('Delimiters:', TUPLE_DELIM, COMPLETE_DELIM)


Delimiters: <|#|> <|COMPLETE|>


## 3) LLM local + fix warnings (`sanitize_extraction_output`)

### 3.1 — LLM local Ollama + fonctions de sanitization

Définition des deux fonctions critiques :
1. **`local_llm_func()`** : Appelle le LLM Ollama en local via endpoint `/api/chat`, avec support des extractions structurées (détection via system prompt) et sanitization automatique
2. **`embedding_func()`** : Génère les embeddings via Ollama `/api/embeddings` (nomic-embed-text, 768 dims)
3. **`sanitize_extraction_output()`** : Nettoie les triplets LLM en supprimant les entités mauvaises, validant les relations, et normalisant les noms (canonicalization)
4. **`canonicalize_entity_name()`** : Standardise les noms d'entités (ex: "LLM" → "Large Language Models")

Ces fonctions assurent la qualité des extractions et réduisent les hallucinations du LLM.

In [4]:
CURRENT_DOC_ID = None

def is_extraction_call(system_prompt):
    if not system_prompt:
        return False
    sp = system_prompt.lower()
    return ('knowledge graph specialist' in sp) or (TUPLE_DELIM in system_prompt)

import re

CANONICAL_ALIASES = {
    "llm": "Large Language Models",
    "llms": "Large Language Models",
    "large language model": "Large Language Models",
    "large language models": "Large Language Models",
    "large language models llms": "Large Language Models",
    "llm large language model": "Large Language Models",
    "vlm": "Vision-Language Models",
    "vlms": "Vision-Language Models",
    "vision language model": "Vision-Language Models",
    "vision language models": "Vision-Language Models",
    "vision language models vlms": "Vision-Language Models",
    "gnn": "Graph Neural Networks",
    "gnns": "Graph Neural Networks",
    "graph neural network": "Graph Neural Networks",
    "graph neural networks": "Graph Neural Networks",
    "rag": "Retrieval-Augmented Generation",
    "retrieval augmented generation": "Retrieval-Augmented Generation",
    "retrieval augmented generation rag": "Retrieval-Augmented Generation",
    "transformer architecture": "Transformer Architecture",
    "kv cache": "KV Cache",
    "kv-cache": "KV Cache",
    "gpt 5 mini": "GPT-5-Mini",
    "gpt-5-mini": "GPT-5-Mini",
    "gpt 5 4": "GPT-5.4",
    "gpt-5.4": "GPT-5.4",
    "question answering qa": "Question Answering",
    "llm as a judge": "LLM-as-a-Judge",
    "llm-as-a-judge": "LLM-as-a-Judge",
}

BAD_ENTITY_NAMES = {
    "model", "models", "method", "methods", "dataset", "datasets",
    "task", "tasks", "metric", "metrics", "paper", "study",
    "approach", "system", "framework", "technique", "result", "results"
}

PROTECTED_ACRONYMS = {
    "AI", "ML", "NLP", "LLM", "VLM", "GNN", "RAG", "GPT", "BERT", "T5",
    "CNN", "RNN", "LSTM", "PPO", "SFT", "RLHF", "DPO", "GQA", "MLA", "KV"
}

def normalize_key(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(name or "").lower()).strip()

def canonicalize_entity_name(name: str) -> str:
    s = str(name or "").strip().strip("\"'`")
    s = re.sub(r"\\+", "", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*[-??]\s*", "-", s)
    key = normalize_key(s)

    if key in CANONICAL_ALIASES:
        return CANONICAL_ALIASES[key]

    if re.search(r"\b(GPT|LLM|VLM|GNN|RAG|BERT|T5|AI|ML|NLP|CNN|RNN|LSTM|RLHF|DPO)\b", s, re.I):
        return s

    if s.upper() in PROTECTED_ACRONYMS:
        return s.upper()

    return s.title()

def is_bad_entity(name: str) -> bool:
    key = normalize_key(name)
    words = re.findall(r"[A-Za-z0-9]+", name)

    if key in BAD_ENTITY_NAMES:
        return True

    # Full paper titles create one-node-per-paper fragments. Keep long names only if they look like reusable artifacts.
    if len(words) > 14:
        keep_terms = ["model", "dataset", "benchmark", "architecture", "framework", "algorithm", "tool", "corpus"]
        if not any(t in key for t in keep_terms):
            return True

    return False

def sanitize_extraction_output(text):
    lines = [ln.strip() for ln in (text or "").splitlines() if ln.strip()]
    entities = {}
    relations = []

    for ln in lines:
        if ln == COMPLETE_DELIM:
            break

        parts = ln.split(TUPLE_DELIM)
        if not parts:
            continue

        if parts[0] == "entity":
            if len(parts) == 3:
                parts = parts + [""]
            if len(parts) < 4:
                continue

            name = canonicalize_entity_name(parts[1])
            if is_bad_entity(name):
                continue

            ent_type = parts[2].strip() or "Concept"
            desc = parts[3].strip()
            entities[name] = (ent_type, desc)

        elif parts[0] == "relation":
            if len(parts) == 4:
                parts = parts + [""]
            if len(parts) < 5:
                continue

            src = canonicalize_entity_name(parts[1])
            tgt = canonicalize_entity_name(parts[2])

            if src == tgt or is_bad_entity(src) or is_bad_entity(tgt):
                continue

            pred = parts[3].strip() or "associated_with"
            desc = parts[4].strip()
            relations.append((src, tgt, pred, desc))

    used_entities = {s for s, _, _, _ in relations} | {t for _, t, _, _ in relations}

    fixed = []
    for name in sorted(used_entities):
        ent_type, desc = entities.get(
            name,
            ("Concept", f"{name} is mentioned in a relation extracted from the chunk."),
        )
        fixed.append(f"entity{TUPLE_DELIM}{name}{TUPLE_DELIM}{ent_type}{TUPLE_DELIM}{desc}")

    seen_rel = set()
    for src, tgt, pred, desc in relations:
        key = tuple(sorted([src, tgt])) + (pred.lower(),)
        if key in seen_rel:
            continue
        seen_rel.add(key)
        fixed.append(f"relation{TUPLE_DELIM}{src}{TUPLE_DELIM}{tgt}{TUPLE_DELIM}{pred}{TUPLE_DELIM}{desc}")

    fixed.append(COMPLETE_DELIM)
    return "\n".join(fixed)


async def local_llm_func(prompt, system_prompt=None,
                          history_messages=[], **kwargs):
    """
    LLM local via Ollama — endpoint /api/chat uniquement.
    Version Ollama 0.22.0 ne supporte PAS /v1/chat/completions.
    """
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    for msg in history_messages:
        messages.append(msg)
    messages.append({'role': 'user', 'content': prompt})

    extraction  = is_extraction_call(system_prompt)
    num_predict = MAX_TOKENS_EXTRACT if extraction else MAX_TOKENS_QUERY

    # ── Un seul endpoint : /api/chat ────────────────────────
    try:
        r = requests.post(
            f'{OLLAMA_URL}/api/chat',
            json={
                'model'   : MODEL_NAME,
                'messages': messages,
                'stream'  : False,
                'options' : {
                    'temperature' : 0,
                    'num_predict' : num_predict,
                    'num_ctx'     : OLLAMA_NUM_CTX,
                },
            },
            timeout=600,
        )
        r.raise_for_status()
        out = r.json().get('message', {}).get('content', '') or ''

    except requests.exceptions.ConnectionError:
        print(' Ollama inaccessible — lance : ollama serve')
        raise
    except requests.exceptions.HTTPError as e:
        print(f' Erreur HTTP Ollama : {e}')
        raise

    # ── Logger les triplets si appel d'extraction ───────────
    if extraction:
        out = sanitize_extraction_output(out)
        triplet_logger.parse_extraction(out, doc_id=CURRENT_DOC_ID)

    return out


async def embedding_func(texts):
    """
    Embedding Ollama — endpoint /api/embeddings uniquement.
    Retourne numpy 2D (n_textes, EMBED_DIM).
    """
    embs = []
    for t in texts:
        r = requests.post(
            f'{OLLAMA_URL}/api/embeddings',
            json={'model': EMBED_MODEL, 'prompt': t},
            timeout=300,
        )
        r.raise_for_status()
        emb = r.json().get('embedding', [])
        embs.append(np.array(emb, dtype=np.float32))
    return np.vstack(embs)


print(' LLM local défini — endpoint /api/chat uniquement')
print(f'   Ollama URL  : {OLLAMA_URL}')
print(f'   Modèle LLM  : {MODEL_NAME}')
print(f'   Embedding   : {EMBED_MODEL} ({EMBED_DIM} dim)')

 LLM local défini — endpoint /api/chat uniquement
   Ollama URL  : http://localhost:11434
   Modèle LLM  : llama3.1:8b
   Embedding   : nomic-embed-text (768 dim)


### 3.2 — Test de connectivité Ollama

Vérification que le serveur Ollama est accessible et répond correctement à une requête simple. Ce test diagnostique précoce prévient les blocages lors de l'indexation complète.

In [4]:
import requests
import json

print('Test /api/chat...')

payload = {
    'model': MODEL_NAME,
    'messages': [
        {'role': 'user', 'content': 'Say only: OK'}
    ],
    'stream': False,
    'options': {
        'temperature': 0,
        'num_predict': 8,
        'num_ctx': 2048,
    },
}

try:
    r = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json=payload,
        timeout=600,
    )
    r.raise_for_status()
    data = r.json()
    print('STATUS:', r.status_code)
    print('MESSAGE:', data.get('message', {}).get('content', data))
except requests.exceptions.ReadTimeout as e:
    raise TimeoutError(
        'Ollama a depasse 600s. Verifie que ollama serve est lance, que le modele est deja pull, et que le PC n est pas sature.'
    ) from e


Test /api/chat...
STATUS: 200
MESSAGE: OK


## 4) Indexation 500 docs + checkpoint

### 4.1 — Indexation complète du corpus (500 documents)

Pipeline d'indexation LightRAG avec support du checkpointing pour reprendre après interruption. Étapes :
1. Initialisation du RAG avec embeddings Ollama
2. Lecture du corpus arXiv (500 documents)
3. Insertion progressive avec logging des erreurs
4. Sauvegarde du checkpoint après chaque document
5. Export des triplets extraits en JSON

Durée estimée : 3-4 jours selon la puissance du LLM local.

In [18]:
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc

def load_checkpoint(path: Path):
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {'last_index': 0, 'success': 0, 'errors': []}

def save_checkpoint(path: Path, state: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

async def build_rag(working_dir: Path) -> LightRAG:
    embed = EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=8192,
        model_name=EMBED_MODEL,
        func=embedding_func,
    )
    rag = LightRAG(
        working_dir=str(working_dir),
        llm_model_func=local_llm_func,
        embedding_func=embed,
        llm_model_max_async=LLM_MAX_ASYNC,
        max_parallel_insert=MAX_PARALLEL_INSERT,
        max_extract_input_tokens=MAX_EXTRACT_INPUT_TOKENS,
        entity_extract_max_gleaning=ENTITY_EXTRACT_MAX_GLEANING,
        chunk_token_size=CHUNK_TOKEN_SIZE,
        chunk_overlap_token_size=CHUNK_OVERLAP,
        addon_params={
            'language': 'English',
            'entity_types': [
                'Model', 'Method', 'Dataset', 'Task', 'Metric',
                'Benchmark', 'Architecture', 'Application',
                'Concept', 'Tool', 'Organization'
            ]
        }
    )
    await rag.initialize_storages()
    return rag

async def index_corpus():
    global CURRENT_DOC_ID, triplet_logger
    reset = input(f"Supprimer l'index {INDEX_DIR} et recommencer ? (o/N): ").strip().lower() == 'o'
    if reset:
        if INDEX_DIR.exists():
            shutil.rmtree(INDEX_DIR)
        if CHECKPOINT.exists():
            CHECKPOINT.unlink()
        INDEX_DIR.mkdir(parents=True, exist_ok=True)
        triplet_logger = TripletLogger()

    rag = await build_rag(INDEX_DIR)

    corpus_path = DATA_DIR / 'arxiv_cleaned.json'
    with open(corpus_path, 'r', encoding='utf-8') as f:
        docs_all = json.load(f)
    docs = docs_all[:NB_DOCS]

    state = load_checkpoint(CHECKPOINT)
    start = state['last_index']
    success = state['success']
    errors = state['errors']

    print(f'Start indexation @ {start} / {len(docs)}')

    for i in tqdm(range(start, len(docs)), desc='Indexation'):
        d = docs[i]
        doc_id = d.get('id', str(i))
        text = d.get('text') or f"Title: {d.get('title','')}\n\nAbstract: {d.get('abstract','')}"
        CURRENT_DOC_ID = doc_id
        try:
            await rag.ainsert(text, ids=doc_id, file_paths=doc_id)
            success += 1
        except Exception as e:
            errors.append({'index': i, 'doc_id': doc_id, 'error': str(e)[:300]})
            await asyncio.sleep(2)
        finally:
            save_checkpoint(CHECKPOINT, {'last_index': i + 1, 'success': success, 'errors': errors})
        await asyncio.sleep(SLEEP_BETWEEN_DOCS_S)

    print('DONE. success=', success, 'errors=', len(errors))
    out_path = OUT_DIR / 'triplets_extraction_llm_500_NV.json'
    if triplet_logger.items:
        triplet_logger.save(out_path)
    else:
        print("Aucun nouveau triplet logg?: je n'ecrase pas", out_path)
    return rag

rag = asyncio.run(index_corpus())


INFO: [] Loaded graph from indexes\lightrag_500_connected_v2\graph_chunk_entity_relation.graphml with 5065 nodes, 5068 edges
INFO:nano-vectordb:Load (5065, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': 'indexes\\lightrag_500_connected_v2\\vdb_entities.json'} 5065 data
INFO:nano-vectordb:Load (5068, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': 'indexes\\lightrag_500_connected_v2\\vdb_relationships.json'} 5068 data
INFO:nano-vectordb:Load (500, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': 'indexes\\lightrag_500_connected_v2\\vdb_chunks.json'} 500 data


Start indexation @ 500 / 500


Indexation: 0it [00:00, ?it/s]

DONE. success= 499 errors= 0
Aucun nouveau triplet logg?: je n'ecrase pas data\processed\triplets\triplets_extraction_llm_500_NV.json


### Récupération des triplets LightRAG depuis l'index v2 sans ré-indexation


### 4.2 — Récupération des triplets depuis l'index GraphML

Extraction des données d'indexation LightRAG déjà effectuée (sans re-indexation). Utilise les fichiers persistants :
- `kv_store_full_relations.json` : relations par document
- `kv_store_full_entities.json` : entités par document  
- `graph_chunk_entity_relation.graphml` : graphe complet

Génère un fichier JSON `triplets_extraction_llm_500_NV_recovered.json` compatible avec le format d'export pour analyse ultérieure.  
Cette étape a été ajoutée suite à une interruption du système. Après l'arrêt du PC, j'ai voulu vérifier que l'indexation s'était déroulée correctement avant le bug, mais j'ai accidentellement écrasé le fichier de résultats avec un fichier vide. Cette récupération des triplets depuis l'index GraphML permet donc de restaurer les données sans avoir à ré-indexer le corpus complet.

In [7]:
import json
import networkx as nx
from pathlib import Path

RECOVERY_INDEX_DIR = BASE_DIR / "indexes" / "lightrag_500_connected_v2"
REL_PATH = RECOVERY_INDEX_DIR / "kv_store_full_relations.json"
ENT_PATH = RECOVERY_INDEX_DIR / "kv_store_full_entities.json"
GRAPHML_PATH = RECOVERY_INDEX_DIR / "graph_chunk_entity_relation.graphml"
OUT_RECOVERED = OUT_DIR / "triplets_extraction_llm_500_NV_recovered.json"

if not REL_PATH.exists():
    raise FileNotFoundError(f"Relations introuvables: {REL_PATH}")
if not ENT_PATH.exists():
    raise FileNotFoundError(f"Entit?s introuvables: {ENT_PATH}")
if not GRAPHML_PATH.exists():
    raise FileNotFoundError(f"GraphML introuvable: {GRAPHML_PATH}")

relations_store = json.loads(REL_PATH.read_text(encoding="utf-8"))
entities_store = json.loads(ENT_PATH.read_text(encoding="utf-8"))
g = nx.read_graphml(str(GRAPHML_PATH))

edge_lookup = {}
for u, v, attrs in g.edges(data=True):
    edge_lookup[(str(u), str(v))] = attrs
    edge_lookup[(str(v), str(u))] = attrs

items = []
total_entities = 0
total_relations = 0

for doc_id, rel_data in relations_store.items():
    entity_names = entities_store.get(doc_id, {}).get("entity_names", [])
    relation_pairs = rel_data.get("relation_pairs", [])

    entities = [
        {
            "name": e,
            "type": "",
            "description": ""
        }
        for e in entity_names
    ]

    relations = []
    for src, tgt in relation_pairs:
        attrs = edge_lookup.get((src, tgt), {})
        relations.append({
            "source": src,
            "target": tgt,
            "relation_keywords": attrs.get("keywords", "related_to"),
            "description": attrs.get("description", "")
        })

    total_entities += len(entities)
    total_relations += len(relations)

    items.append({
        "doc_id": doc_id,
        "entities": entities,
        "relations": relations
    })

recovered = {
    "metadata": {
        "source": "recovered_from_lightrag_500_connected_v2",
        "total_documents": len(items),
        "total_entities": total_entities,
        "total_relations": total_relations,
        "index_dir": str(RECOVERY_INDEX_DIR),
        "graphml": str(GRAPHML_PATH)
    },
    "triplets": items
}

OUT_RECOVERED.write_text(
    json.dumps(recovered, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Recovered saved:", OUT_RECOVERED)
print("Documents:", len(items))
print("Entities:", total_entities)
print("Relations:", total_relations)


Recovered saved: data\processed\triplets\triplets_extraction_llm_500_NV_recovered.json
Documents: 500
Entities: 5433
Relations: 5069


In [19]:
import networkx as nx
import numpy as np
from pathlib import Path

GRAPHML_PATH = INDEX_DIR / "graph_chunk_entity_relation.graphml"

if not GRAPHML_PATH.exists():
    print("GraphML introuvable :", GRAPHML_PATH)
else:
    g = nx.read_graphml(str(GRAPHML_PATH))

    n_nodes = g.number_of_nodes()
    n_edges = g.number_of_edges()
    degrees = [d for _, d in g.degree()]
    deg_mean = float(np.mean(degrees)) if degrees else 0.0

    print("=== RAW LIGHTRAG GRAPH - Densité du graphe ===")
    print("Nodes (entités) :", n_nodes)
    print("Edges (relations):", n_edges)
    print("Degré moyen      :", round(deg_mean, 2), "(cible > 3)")
    print("Degré max        :", max(degrees) if degrees else 0)
    print("Densité (nx)     :", round(nx.density(g), 6))

    print("\nTop 10 hubs (entités les plus connectées):")
    top = sorted(g.degree(), key=lambda x: x[1], reverse=True)[:10]
    for node, deg in top:
        label = g.nodes[node].get("name") or g.nodes[node].get("entity_name") or str(node)
        print(f"  {deg:>3}  {label[:80]}")

=== RAW LIGHTRAG GRAPH - Densité du graphe ===
Nodes (entités) : 5065
Edges (relations): 5068
Degré moyen      : 2.0 (cible > 3)
Degré max        : 215
Densité (nx)     : 0.000395

Top 10 hubs (entités les plus connectées):
  215  Large Language Models
   41  Vision-Language Models
   28  Agentic AI Architecture
   27  Stellar-E
   27  Solartformer
   23  Gsar
   22  Physnote
   22  Cuda Tile
   22  Memanto
   21  Code Broker


## 5) Évaluation quantitative (latence + nodes traversed approx + RAGAS faithfulness)

### 5.1 — Évaluation quantitative du mode Hybrid (latence + nodes + RAGAS Faithfulness)

Benchmark complet du LightRAG en mode hybrid sur 3 questions de test. Mesures :
- **Latence** : temps de réponse par query
- **Nodes traversed** : nombre approximatif d'entités consultées
- **Contexts** : chunks récupérés
- **RAGAS Faithfulness** : pourcentage de la réponse fidèle aux contextes (via LLM Ollama)

Résultats sauvegardés en JSON pour analyse comparative.

In [8]:
QUESTIONS = [
    'What are the main methods used in machine learning for classification?',
    'How do graph neural networks improve knowledge representation?',
    'What are the limitations of transformer models in NLP?',
]

def count_traversed_nodes(result_obj) -> int:
    try:
        ro = getattr(result_obj, 'retriever_output', None)
        if ro is None and isinstance(result_obj, dict):
            ro = result_obj.get('retriever_output')
        if not ro:
            return 0
        if isinstance(ro, dict):
            nodes = ro.get('nodes') or ro.get('entity_nodes') or ro.get('entities')
            if isinstance(nodes, list):
                return len({str(x) for x in nodes})
        return 0
    except Exception:
        return 0

async def run_benchmark(rag):
    rows = []
    for q in QUESTIONS:
        t0 = time.time()
        res = await rag.aquery(q, param=QueryParam(mode='hybrid', top_k=40, chunk_top_k=20, enable_rerank=False))
        lat = round(time.time() - t0, 2)
        ctx = await rag.aquery(q, param=QueryParam(mode='hybrid', only_need_context=True, top_k=40, chunk_top_k=20, enable_rerank=False))
        rows.append({
            'question': q,
            'answer': str(res),
            'latency_s': lat,
            'nodes_traversed_approx': count_traversed_nodes(res),
            'contexts': ctx if isinstance(ctx, list) else [str(ctx)]
        })
        print('latency=', lat, 's |', q[:55])
    with open(OUT_DIR / 'benchmark_hybrid_results.json', 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    return rows

bench_rows = asyncio.run(run_benchmark(rag))

try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import faithfulness
    from langchain_community.llms.ollama import Ollama

    ragas_ds = Dataset.from_list([
        {'question': r['question'], 'answer': r['answer'], 'contexts': r['contexts']}
        for r in bench_rows
    ])
    llm_eval = Ollama(model=MODEL_NAME, base_url=OLLAMA_URL)
    scores = evaluate(ragas_ds, metrics=[faithfulness], llm=llm_eval)
    print(scores)
    scores.to_pandas().to_json(
        OUT_DIR / 'ragas_faithfulness.json',
        orient='records',
        force_ascii=False,
        indent=2
    )
except ModuleNotFoundError as e:
    print('Benchmark hybrid sauvegarde dans:', OUT_DIR / 'benchmark_hybrid_results.json')
    print('RAGAS non execute a cause d une dependance manquante:', e)
    print('Relance la cellule des dependances, puis relance cette cellule.')


INFO: LLM func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: hybrid:keywords:d934ede02cfa2dbf5175e0c281e5b2d2
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO: Query nodes: Supervised learning, Neural networks, Support vector machines, Decision trees (top_k:40, cosine:0.2)
INFO: Query edges: Machine learning, Classification (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.
INFO: Query nodes: Supervised learning, Neural networks, Support vector machines, Decision trees (top_k:40, cosine:0.2)
INFO: Query edges: Machine learning, Classification (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.


latency= 39.32 s | What are the main methods used in machine learning for 


INFO:  == LLM cache == saving: hybrid:keywords:bab424bc04eb24b56b7dabae879e2feb
INFO: Query nodes: Deep learning models, Neural network architectures, Node embeddings, Graph attention mechanisms (top_k:40, cosine:0.2)
INFO: Query edges: Graph neural networks, Knowledge representation (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.
INFO: Query nodes: Deep learning models, Neural network architectures, Node embeddings, Graph attention mechanisms (top_k:40, cosine:0.2)
INFO: Query edges: Graph neural networks, Knowledge representation (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.


latency= 34.82 s | How do graph neural networks improve knowledge represen


INFO:  == LLM cache == saving: hybrid:keywords:461f94abeca34ed9eb76a5752ec46849
INFO: Query nodes: Attention mechanism, Self-supervised learning, Pre-training, Fine-tuning (top_k:40, cosine:0.2)
INFO: Query edges: Transformer models, NLP limitations (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.
INFO: Query nodes: Attention mechanism, Self-supervised learning, Pre-training, Fine-tuning (top_k:40, cosine:0.2)
INFO: Query edges: Transformer models, NLP limitations (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.


latency= 30.32 s | What are the limitations of transformer models in NLP?


c:\Users\ADMIN\Desktop\PFE_Agentic_Graphrag\GraphRag_PFE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_39632\1923246046.py:46: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_39632\1923246046.py:53: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  ll

{'faithfulness': 0.7778}


## 6) Export Neo4j (GraphML → batch UNWIND)

### 6.1 — Export Neo4j (GraphML → Batch UNWIND)

Conversion du graphe LightRAG depuis le format GraphML vers Neo4j en utilisant les opérations batch avec UNWIND (performance optimale). Étapes :
1. Lecture du GraphML
2. Création des contraintes (Entity.id UNIQUE)
3. Nettoyage optionnel des entités anciennes
4. Insertion batch des nœuds (2000 par batch)
5. Insertion batch des relations RELATES_TO

Cette approche batch évite les timeout et les surcharges réseau.

In [18]:
import networkx as nx
from neo4j import GraphDatabase

GRAPHML_PATH = INDEX_DIR / 'graph_chunk_entity_relation.graphml'

def pick_relation_label(attrs: dict) -> str:
    keys = ['relationship_keywords','relation','predicate','type','label','keywords','relationship','edge_type']
    for k in keys:
        v = attrs.get(k)
        if v is not None and str(v).strip():
            return str(v).strip()
    return 'unknown'

def export_graphml_to_neo4j(graphml_path: Path, batch_size=2000, clear_only_entities=True):
    if not (NEO4J_URI and NEO4J_PASSWORD):
        raise ValueError('Set NEO4J_URI + NEO4J_PASSWORD dans .env')
    g = nx.read_graphml(str(graphml_path))

    def n_name(n):
        a = g.nodes[n]
        return a.get('name') or a.get('entity_name') or str(n)
    def n_type(n):
        a = g.nodes[n]
        return a.get('type') or a.get('entity_type') or ''
    def n_desc(n):
        a = g.nodes[n]
        return (a.get('description') or a.get('entity_description') or '')[:500]
    def e_desc(a):
        return (a.get('relationship_description') or a.get('description') or '')[:500]

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        session.run('CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.id IS UNIQUE')
        if clear_only_entities:
            session.run('MATCH (n:Entity) DETACH DELETE n')

        nodes_rows = [{'id': str(n), 'name': n_name(n), 'type': n_type(n), 'description': n_desc(n)} for n in g.nodes()]
        cypher_nodes = """
        UNWIND $rows AS row
        MERGE (e:Entity {id: row.id})
        SET e.name = row.name,
            e.type = row.type,
            e.description = row.description
        """
        for i in tqdm(range(0, len(nodes_rows), batch_size), desc='Neo4j nodes'):
            session.run(cypher_nodes, rows=nodes_rows[i:i+batch_size])

        edges_rows = []
        for u, v, attrs in g.edges(data=True):
            edges_rows.append({'src': str(u), 'tgt': str(v), 'type': pick_relation_label(attrs), 'description': e_desc(attrs)})

        cypher_edges = """
        UNWIND $rows AS row
        MATCH (a:Entity {id: row.src})
        MATCH (b:Entity {id: row.tgt})
        MERGE (a)-[r:RELATES_TO {type: row.type}]->(b)
        SET r.description = row.description
        """
        for i in tqdm(range(0, len(edges_rows), batch_size), desc='Neo4j edges'):
            session.run(cypher_edges, rows=edges_rows[i:i+batch_size])

    driver.close()
    print('Neo4j import DONE:', g.number_of_nodes(), 'nodes |', g.number_of_edges(), 'edges')

export_graphml_to_neo4j(GRAPHML_PATH)


Neo4j edges: 100%|██████████| 3/3 [00:00<00:00,  3.68it/s]

Neo4j import DONE: 5065 nodes | 5068 edges


### 6.2 — Requêtes exploratives Neo4j (10 analyses du graphe)

Exécution de 10 requêtes Cypher pour caractériser le graphe de connaissances :
1. Top entités connectées (degré)
2. Types de relations fréquents
3. Entités isolées
4. Paires d'entités les plus reliées
5. Chemins court LLM <-> GNN
6. Sous-graphe thématique LLM
7. Sous-graphe thématique RAG
8. Sous-graphe thématique GNN
9. Statistiques de degré (moyen, max, min)
10. Détection de communautés avec Louvain 

Résultats exportés en JSON pour visualisation et analyse.

In [ ]:
from neo4j import GraphDatabase
import json

NEO4J_EXPLORATION_OUT = OUT_DIR / "neo4j_exploratory_queries_results.json"

cypher_queries = {
    "01_top_connected_entities": """
    MATCH (e:Entity)-[r]-()
    RETURN e.name AS entity, e.type AS type, count(r) AS degree
    ORDER BY degree DESC
    LIMIT 20
    """,

    "02_relation_type_counts": """
    MATCH ()-[r:RELATES_TO]->()
    RETURN r.type AS relation_type, count(*) AS count
    ORDER BY count DESC
    LIMIT 30
    """,

    "03_isolated_entities": """
    MATCH (e:Entity)
    WHERE NOT (e)--()
    RETURN e.name AS entity, e.type AS type
    LIMIT 50
    """,

    "04_top_entity_pairs": """
    MATCH (a:Entity)-[r:RELATES_TO]->(b:Entity)
    RETURN a.name AS source, b.name AS target, r.type AS relation, count(*) AS count
    ORDER BY count DESC
    LIMIT 30
    """,

    "05_shortest_path_llm_gnn": """
    MATCH (a:Entity), (b:Entity)
    WHERE toLower(a.name) CONTAINS 'language model'
      AND toLower(b.name) CONTAINS 'graph neural'
    MATCH p = shortestPath((a)-[:RELATES_TO*..5]-(b))
    RETURN [n IN nodes(p) | n.name] AS path, length(p) AS hops
    LIMIT 10
    """,

    "06_llm_thematic_subgraph": """
    MATCH (a:Entity)-[r:RELATES_TO]-(b:Entity)
    WHERE toLower(a.name) CONTAINS 'llm'
       OR toLower(a.name) CONTAINS 'language model'
       OR toLower(b.name) CONTAINS 'llm'
       OR toLower(b.name) CONTAINS 'language model'
    RETURN a.name AS source, r.type AS relation, b.name AS target
    LIMIT 80
    """,

    "07_rag_thematic_subgraph": """
    MATCH (a:Entity)-[r:RELATES_TO]-(b:Entity)
    WHERE toLower(a.name) CONTAINS 'retrieval'
       OR toLower(a.name) CONTAINS 'rag'
       OR toLower(b.name) CONTAINS 'retrieval'
       OR toLower(b.name) CONTAINS 'rag'
    RETURN a.name AS source, r.type AS relation, b.name AS target
    LIMIT 80
    """,

    "08_gnn_thematic_subgraph": """
    MATCH (a:Entity)-[r:RELATES_TO]-(b:Entity)
    WHERE toLower(a.name) CONTAINS 'graph neural'
       OR toLower(b.name) CONTAINS 'graph neural'
    RETURN a.name AS source, r.type AS relation, b.name AS target
    LIMIT 80
    """,

    "09_graph_degree_summary": """
    MATCH (e:Entity)
    OPTIONAL MATCH (e)-[r]-()
    WITH e, count(r) AS degree
    RETURN count(e) AS nodes,
           avg(degree) AS avg_degree,
           max(degree) AS max_degree,
           min(degree) AS min_degree
    """,

    "10_community_detection_gds": """
    CALL gds.graph.project(
      'entityGraph',
      'Entity',
      {RELATES_TO: {orientation: 'UNDIRECTED'}
    )
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount
    """
}

def run_neo4j_exploration():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    results = {}

    with driver.session() as session:
        for name, query in cypher_queries.items():
            try:
                rows = [dict(r) for r in session.run(query)]
                results[name] = rows
                print(name, "OK", len(rows), "rows")
            except Exception as e:
                results[name] = {"error": str(e)}
                print(name, "ERROR:", str(e)[:200])

        try:
            community_query = """
            CALL gds.louvain.stream('entityGraph')
            YIELD nodeId, communityId
            WITH communityId, collect(gds.util.asNode(nodeId).name)[0..15] AS sample_entities, count(*) AS size
            RETURN communityId, size, sample_entities
            ORDER BY size DESC
            LIMIT 10
            """
            results["10b_louvain_communities"] = [dict(r) for r in session.run(community_query)]
            print("10b_louvain_communities OK")
        except Exception as e:
            results["10b_louvain_communities"] = {"error": str(e)}
            print("GDS Louvain ERROR:", str(e)[:200])

    driver.close()

    with open(NEO4J_EXPLORATION_OUT, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("Saved:", NEO4J_EXPLORATION_OUT)
    return results

neo4j_results = run_neo4j_exploration()

01_top_connected_entities OK 20 rows
02_relation_type_counts OK 30 rows
03_isolated_entities OK 28 rows
04_top_entity_pairs OK 30 rows


INFO:neo4j.notifications:Received notification from DBMS server: <GqlStatusObject gql_status='03N90', status_description="info: cartesian product. The disconnected pattern '(a:Entity), (b:Entity)' builds a cartesian product. A cartesian product may produce a large amount of data and slow down query processing.", position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='PERFORMANCE', classification=<NotificationClassification.PERFORMANCE: 'PERFORMANCE'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'PERFORMANCE', '_severity': 'INFORMATION', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    MATCH (a:Entity), (b:Entity)\n    WHERE toLower(a.name) CONTAINS 'language model'\n      AND toLower(b.name) CONTAINS 'graph neural'\n    MATCH p = shortestPath((a)-[:RELATES_TO*..5]-(b))\n    RETURN [n IN nodes(p) | 

05_shortest_path_llm_gnn OK 3 rows
06_llm_thematic_subgraph OK 80 rows
07_rag_thematic_subgraph OK 80 rows
08_gnn_thematic_subgraph OK 28 rows
09_graph_degree_summary OK 1 rows
10_community_detection_gds ERROR: {neo4j_code: Neo.ClientError.Procedure.ProcedureNotFound} {message: There is no procedure with the name `gds.graph.project` registered for this database instance. Please ensure you've spelled the proc
GDS Louvain ERROR: {neo4j_code: Neo.ClientError.Procedure.ProcedureNotFound} {message: There is no procedure with the name `gds.louvain.stream` registered for this database instance. Please ensure you've spelled the pro
Saved: data\processed\triplets\neo4j_exploratory_queries_results.json


## 7) Comparaison entre triplets

### 7.1 — Extraction NER + RE avec spaCy et REBEL (rappel++)

Pipeline complet d'extraction alternatif basé sur modèles entraînés (pas LLM) :
- **spaCy** : segmentation offline des phrases (sentencizer)
- **REBEL** : extraction des triplets (head, relation, tail) en seq2seq
- **Fenêtres de phrases** : technique de rappel++ pour couvrir plus de contexte (window=3 phrases, step=2)
- **Parsing strict** : normalisation, dédup locale, filtrage entités trop courtes
- **Checkpoint** : sauvegarde tous les 25 documents

Avantages vs LightRAG : pas de dépendance au LLM, extraction plus stable, relations propres (instance_of, subclass_of, use, etc.)
Durée : ~1.1 heure pour 500 documents.

In [ ]:
import os
import re
import json
import time
import warnings
from pathlib import Path
from collections import Counter

import torch
from tqdm import tqdm
import spacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_FILE = Path("data/processed/arxiv_cleaned.json")
OUTPUT_DIR = Path("data/processed/triplets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "rebel_triplets_entities_recallpp.json"
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint_rebel_recallpp.json"
SAVE_EVERY = 25
MAX_DOCS = None  # None = tout le corpus

REBEL_MODEL_NAME = "Babelscape/rebel-large"

# REBEL (plus riche)
MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 512
NUM_BEAMS = 5  # un peu plus que 4 => parfois + de triplets (mais plus lent)

# Fenêtres de phrases (rappel++)
WINDOW_SENTENCES = 3      # fenêtre de 3 phrases (bon compromis contexte/rappel)
STEP_SENTENCES = 2        # avance de 2 => overlap 1 phrase
MIN_SENT_CHARS = 15       # phrases plus courtes acceptées

# Normalisation
LOWER_RELATION = True
MIN_ENTITY_LEN = 3

# réduire warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore", category=FutureWarning)

_ws = re.compile(r"\s+")


def clean_text(s: str) -> str:
    s = (s or "").replace("\n", " ").strip()
    return _ws.sub(" ", s)


def norm_ent(e: str) -> str:
    e = clean_text(e)
    return e.strip(" ,.;:()[]{}\"'")


def norm_rel(r: str) -> str:
    r = clean_text(r)
    return r.lower() if LOWER_RELATION else r


def parse_rebel(decoded_text: str):
    """
    Parser REBEL (conforme README Babelscape) :
      <triplet> SUBJECT <subj> OBJECT <obj> RELATION ...
    """
    triplets = []
    subject, relation, object_ = "", "", ""
    current = "x"

    tokens = (
        decoded_text.replace("<s>", "")
        .replace("<pad>", "")
        .replace("</s>", "")
        .split()
    )

    for tok in tokens:
        if tok == "<triplet>":
            current = "t"
            if relation.strip():
                triplets.append({"head": subject.strip(), "type": relation.strip(), "tail": object_.strip()})
            relation, subject = "", ""

        elif tok == "<subj>":
            current = "s"
            if relation.strip():
                triplets.append({"head": subject.strip(), "type": relation.strip(), "tail": object_.strip()})
            object_ = ""

        elif tok == "<obj>":
            current = "o"
            relation = ""

        else:
            if current == "t":
                subject += " " + tok
            elif current == "s":
                object_ += " " + tok
            elif current == "o":
                relation += " " + tok

    if subject.strip() and relation.strip() and object_.strip():
        triplets.append({"head": subject.strip(), "type": relation.strip(), "tail": object_.strip()})

    # normalisation + dédup locale
    seen = set()
    uniq = []
    for t in triplets:
        h = norm_ent(t["head"])
        r = norm_rel(t["type"])
        ta = norm_ent(t["tail"])

        if len(h) < MIN_ENTITY_LEN or len(ta) < MIN_ENTITY_LEN or len(r) < 2:
            continue
        if h.lower() == ta.lower():
            continue

        key = (h, r, ta)
        if key not in seen:
            seen.add(key)
            uniq.append({"head": h, "type": r, "tail": ta})
    return uniq


def rebel_extract_triplets(text_segment: str, tokenizer, model):
    enc = tokenizer(
        text_segment,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.inference_mode():
        gen_ids = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS
        )

    decoded = tokenizer.decode(gen_ids[0], skip_special_tokens=False)
    return parse_rebel(decoded)


def dedup_triplets(triplets):
    seen = set()
    out = []
    for t in triplets:
        key = (t["head"], t["type"], t["tail"])
        if key not in seen:
            seen.add(key)
            out.append(t)
    return out


def derive_entities_from_triplets(triplets):
    """
    Entités = union(head, tail), dédupliqué.
    """
    ents = []
    seen = set()
    for t in triplets:
        for k in ("head", "tail"):
            e = norm_ent(t[k])
            if len(e) < MIN_ENTITY_LEN:
                continue
            key = e.lower()
            if key not in seen:
                seen.add(key)
                ents.append({"text": e})
    return ents


def build_sentence_windows(sentences, tokenizer):
    """
    Fenêtres de phrases avec overlap:
    - window = WINDOW_SENTENCES
    - step   = STEP_SENTENCES
    Chaque fenêtre doit respecter MAX_INPUT_TOKENS (sinon on réduit la fenêtre).
    """
    sents = [clean_text(s) for s in sentences if len(clean_text(s)) >= MIN_SENT_CHARS]
    windows = []
    n = len(sents)
    i = 0

    while i < n:
        # tentative fenêtre de WINDOW_SENTENCES
        j = min(i + WINDOW_SENTENCES, n)
        chunk = " ".join(sents[i:j]).strip()

        # si trop long, réduire jusqu'à rentrer dans MAX_INPUT_TOKENS
        while chunk:
            n_tokens = len(tokenizer(chunk, add_special_tokens=False)["input_ids"])
            if n_tokens <= MAX_INPUT_TOKENS:
                break
            # retirer la dernière phrase
            j -= 1
            if j <= i:
                # si même une phrase est trop longue, on tronque via tokenizer 
                chunk = sents[i]
                break
            chunk = " ".join(sents[i:j]).strip()

        if chunk:
            windows.append(chunk)

        if i + STEP_SENTENCES <= i:
            i += 1
        else:
            i += STEP_SENTENCES

    return windows


def main():
    print(f"Device : {DEVICE}")

    # spaCy offline segmentation
    print("Chargement spaCy (offline sentencizer)...")
    nlp = spacy.blank("en")
    nlp.add_pipe("sentencizer")

    # REBEL
    print("Chargement REBEL...")
    rebel_tokenizer = AutoTokenizer.from_pretrained(REBEL_MODEL_NAME)
    rebel_model = AutoModelForSeq2SeqLM.from_pretrained(REBEL_MODEL_NAME).to(DEVICE)
    if DEVICE == "cuda":
        rebel_model = rebel_model.half()
    rebel_model.eval()

    # Dataset
    print("\nChargement dataset...")
    papers = json.loads(INPUT_FILE.read_text(encoding="utf-8"))
    print(f"{len(papers)} documents chargés\n")
    if MAX_DOCS is not None:
        papers = papers[:MAX_DOCS]

    # Checkpoint
    start_idx = 0
    results = []
    global_rel_counter = Counter()
    parsing_success = 0
    total_triplets = 0
    total_entities = 0

    if CHECKPOINT_FILE.exists():
        ck = json.loads(CHECKPOINT_FILE.read_text(encoding="utf-8"))
        start_idx = ck.get("next_index", 0)
        results = ck.get("results", [])
        meta = ck.get("metadata_partial", {})
        parsing_success = meta.get("parsing_success", 0)
        total_triplets = meta.get("total_triplets", 0)
        total_entities = meta.get("total_entities", 0)
        global_rel_counter = Counter(ck.get("global_rel_counter", {}))
        print(f"[Checkpoint] Reprise à l’index {start_idx} (déjà {len(results)} docs).")

    t0 = time.time()

    for idx in tqdm(range(start_idx, len(papers)), desc="spaCy + REBEL (rappel++)"):
        p = papers[idx]
        title = clean_text(p.get("title", ""))
        abstract = clean_text(p.get("abstract", ""))
        full_text = clean_text(f"{title}. {abstract}")

        # segmentation -> fenêtres
        doc = nlp(full_text)
        sentences = [s.text for s in doc.sents]
        windows = build_sentence_windows(sentences, rebel_tokenizer)

        # REBEL sur chaque fenêtre
        doc_triplets = []
        for w in windows:
            doc_triplets.extend(rebel_extract_triplets(w, rebel_tokenizer, rebel_model))

        doc_triplets = dedup_triplets(doc_triplets)
        doc_entities = derive_entities_from_triplets(doc_triplets)

        if doc_triplets:
            parsing_success += 1

        total_triplets += len(doc_triplets)
        total_entities += len(doc_entities)
        for t in doc_triplets:
            global_rel_counter[t["type"]] += 1

        results.append({
            "id": p.get("id", ""),
            "title": title,
            "entities": doc_entities,
            "triplets": doc_triplets
        })

        if (idx + 1) % SAVE_EVERY == 0:
            ck = {
                "next_index": idx + 1,
                "results": results,
                "global_rel_counter": dict(global_rel_counter),
                "metadata_partial": {
                    "parsing_success": parsing_success,
                    "total_triplets": total_triplets,
                    "total_entities": total_entities
                }
            }
            CHECKPOINT_FILE.write_text(json.dumps(ck, ensure_ascii=False, indent=2), encoding="utf-8")

    nb_docs = len(results)
    taux = (parsing_success / max(nb_docs, 1)) * 100
    runtime = time.time() - t0

    out = {
        "metadata": {
            "nb_documents": nb_docs,
            "parsing_success": parsing_success,
            "taux_parsing_pct": round(taux, 2),
            "total_entities": total_entities,
            "total_triplets": total_triplets,
            "moy_triplets_par_doc": round(total_triplets / max(nb_docs, 1), 3),
            "runtime_sec": round(runtime, 2),
            "top_relations_20": global_rel_counter.most_common(20),
            "params": {
                "MAX_INPUT_TOKENS": MAX_INPUT_TOKENS,
                "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
                "NUM_BEAMS": NUM_BEAMS,
                "WINDOW_SENTENCES": WINDOW_SENTENCES,
                "STEP_SENTENCES": STEP_SENTENCES,
                "MIN_SENT_CHARS": MIN_SENT_CHARS,
                "SPACY_MODE": "blank('en') + sentencizer (offline)"
            }
        },
        "results": results
    }

    OUTPUT_FILE.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

    # Exemples réels
    print("\nExemples (docs les plus riches) :")
    top_docs = sorted(results, key=lambda r: len(r.get("triplets", [])), reverse=True)[:2]
    for d in top_docs:
        print(f"\n- {d['id']} | triplets={len(d['triplets'])} | entities={len(d['entities'])} | {d['title'][:90]}...")
        for t in d["triplets"][:10]:
            print(f"  → [{t['head']}] --{t['type']}--> [{t['tail']}]")

    print(f"\nSauvegardé dans {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

c:\Users\ADMIN\Desktop\PFE_Agentic_Graphrag\GraphRag_PFE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu
Chargement spaCy (offline sentencizer)...
Chargement REBEL...


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 3401.61it/s]



Chargement dataset...
500 documents chargés



spaCy + REBEL (rappel++):   0%|          | 0/500 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=512) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refe


Exemples (docs les plus riches) :

- 2604.23747v1 | triplets=14 | entities=8 | SFT-then-RL Outperforms Mixed-Policy Methods for LLM Reasoning...
  → [SFT-then-RL] --part of--> [LLM]
  → [LLM] --has part--> [SFT-then-RL]
  → [LLM] --has part--> [TRL]
  → [LLM] --has part--> [OpenRLHF]
  → [LLM] --has part--> [Llama-Factory]
  → [TRL] --part of--> [LLM]
  → [OpenRLHF] --part of--> [LLM]
  → [Llama-Factory] --part of--> [LLM]
  → [TRL] --has part--> [LLama-Factory]
  → [Llama-Factory] --part of--> [TRL]

- 2604.22748v1 | triplets=13 | entities=12 | Agentic World Modeling: Foundations, Capabilities, Laws, and Beyond...
  → [AI systems] --has part--> [Agents]
  → [Agents] --part of--> [AI systems]
  → [L1 Predictor] --part of--> [L2 Simulator]
  → [L1 Predictor] --part of--> [L3 Evolver]
  → [L2 Simulator] --has part--> [L1 Predictor]
  → [L2 Simulator] --has part--> [L3 Evolver]
  → [L3 Evolver] --has part--> [L1 Predictor]
  → [L3 Evolver] --part of--> [L2 Simulator]
  → [physical] --ins

### 7.2 — Comparaison qualitative LightRAG vs REBEL

Analyse comparative détaillée des deux méthodes d'extraction :
- **Overlap exact** : triplets identiques
- **Overlap loose** : même paire d'entités (prédicat différent)
- **Top relations** : prédicats les plus fréquents par système
- **Exemples** : triplets exclusifs à chaque système

Génère :
- JSON `comparison_lightrag_vs_rebel_qualitative.json` : résumé stats + exemples
- CSV `manual_validation_50_triplets.csv` : 50 triplets aléatoires (25 LightRAG + 25 REBEL) pour validation manuelle

In [20]:
import json, re, random
import pandas as pd
from collections import Counter

LIGHTRAG_NV_PATH = OUT_DIR / "triplets_extraction_llm_500_NV_recovered.json"
LIGHTRAG_FALLBACK_PATH = OUT_DIR / "triplets_extraction_llm_500.json"
REBEL_PATH = OUT_DIR / "rebel_triplets_entities_recallpp.json"

COMPARISON_OUT = OUT_DIR / "comparison_lightrag_vs_rebel_qualitative.json"
MANUAL_CSV = OUT_DIR / "manual_validation_50_triplets.csv"

random.seed(42)

def norm(x):
    x = str(x or "").lower().strip()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

def key(t, loose=False):
    h = norm(t["head"])
    r = norm(t["relation"])
    ta = norm(t["tail"])
    return (h, ta) if loose else (h, r, ta)

def load_lightrag(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for doc in data.get("triplets", []):
        for r in doc.get("relations", []):
            rows.append({
                "source_system": "LightRAG",
                "doc_id": doc.get("doc_id", ""),
                "head": r.get("source", ""),
                "relation": r.get("relation_keywords", ""),
                "tail": r.get("target", "")
            })
    return rows, data.get("metadata", {})

def load_rebel(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for doc in data.get("results", []):
        for t in doc.get("triplets", []):
            rows.append({
                "source_system": "spaCy_REBEL",
                "doc_id": doc.get("id", ""),
                "head": t.get("head", ""),
                "relation": t.get("type", ""),
                "tail": t.get("tail", "")
            })
    return rows, data.get("metadata", {})

def dedup(rows):
    seen, out = set(), []
    for r in rows:
        k = key(r)
        if not all(k):
            continue
        if k not in seen:
            seen.add(k)
            out.append(r)
    return out

# LightRAG: utiliser NV si rempli, sinon fallback
light_rows, light_meta = load_lightrag(LIGHTRAG_NV_PATH)

if len(light_rows) == 0:
    print("WARNING: triplets_extraction_llm_500_NV.json est vide.")
    print("Utilisation du fichier fallback:", LIGHTRAG_FALLBACK_PATH)
    light_rows, light_meta = load_lightrag(LIGHTRAG_FALLBACK_PATH)

rebel_rows, rebel_meta = load_rebel(REBEL_PATH)

light_rows = dedup(light_rows)
rebel_rows = dedup(rebel_rows)

light_exact = {key(t) for t in light_rows}
rebel_exact = {key(t) for t in rebel_rows}

light_loose = {key(t, loose=True) for t in light_rows}
rebel_loose = {key(t, loose=True) for t in rebel_rows}

summary = {
    "lightrag_unique_triplets": len(light_rows),
    "rebel_unique_triplets": len(rebel_rows),
    "exact_overlap": len(light_exact & rebel_exact),
    "entity_pair_overlap": len(light_loose & rebel_loose),
    "lightrag_only": len(light_exact - rebel_exact),
    "rebel_only": len(rebel_exact - light_exact),
    "lightrag_top_relations": Counter([norm(t["relation"]) for t in light_rows]).most_common(20),
    "rebel_top_relations": Counter([norm(t["relation"]) for t in rebel_rows]).most_common(20),
}

comparison = {
    "summary": summary,
    "examples_lightrag_only": [t for t in light_rows if key(t) not in rebel_exact][:30],
    "examples_rebel_only": [t for t in rebel_rows if key(t) not in light_exact][:30],
    "examples_common": [t for t in light_rows if key(t) in rebel_exact][:30],
}

COMPARISON_OUT.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# 50 triplets à valider manuellement: 25 LightRAG + 25 spaCy_REBEL
sample_light = random.sample(light_rows, min(25, len(light_rows)))
sample_rebel = random.sample(rebel_rows, min(25, len(rebel_rows)))
manual_sample = sample_light + sample_rebel
random.shuffle(manual_sample)

manual_rows = []
for i, t in enumerate(manual_sample, start=1):
    other_exact = key(t) in (rebel_exact if t["source_system"] == "LightRAG" else light_exact)
    other_pair = key(t, loose=True) in (rebel_loose if t["source_system"] == "LightRAG" else light_loose)

    manual_rows.append({
        "validation_id": i,
        "source_system": t["source_system"],
        "doc_id": t["doc_id"],
        "head": t["head"],
        "relation": t["relation"],
        "tail": t["tail"],
        "exists_exactly_in_other_system": other_exact,
        "same_entity_pair_in_other_system": other_pair,
        "manual_is_correct": "",  # remplir: 1 correct, 0 incorrect
        "manual_error_type": "",  # entity_error, relation_error, hallucination, too_vague
        "manual_notes": ""
    })

manual_df = pd.DataFrame(manual_rows)
manual_df.to_csv(MANUAL_CSV, index=False, encoding="utf-8-sig")

print("=== Comparaison qualitative ===")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Comparaison sauvegardée:", COMPARISON_OUT)
print("CSV validation manuelle sauvegardé:", MANUAL_CSV)
print("Remplis manual_is_correct avec 1 ou 0, puis lance la cellule suivante.")
manual_df.head()

=== Comparaison qualitative ===
{
  "lightrag_unique_triplets": 5066,
  "rebel_unique_triplets": 2788,
  "exact_overlap": 0,
  "entity_pair_overlap": 31,
  "lightrag_only": 5066,
  "rebel_only": 2788,
  "lightrag_top_relations": [
    [
      "model application task",
      25
    ],
    [
      "dataset evaluation model performance",
      23
    ],
    [
      "application method",
      22
    ],
    [
      "dataset usage method application",
      19
    ],
    [
      "evaluation performance",
      19
    ],
    [
      "comparison evaluation",
      19
    ],
    [
      "dataset model evaluation",
      18
    ],
    [
      "method application model usage",
      17
    ],
    [
      "framework component knowledge repository",
      17
    ],
    [
      "method application task relevance",
      17
    ],
    [
      "model comparison performance improvement",
      16
    ],
    [
      "dataset usage model application",
      16
    ],
    [
      "benchmarking evaluation

,validation_id,source_system,doc_id,head,relation,tail,exists_exactly_in_other_system,same_entity_pair_in_other_system,manual_is_correct,manual_error_type,manual_notes
0,1,LightRAG,2604.21999v2,Adaptive Computation Time (Act),"accuracy boost,performance comparison",Fixed-Depth Processing,False,False,,,
1,2,LightRAG,2604.21430v1,Cohen'S D,"effect size,metric",Moral Judgements,False,False,,,
2,3,LightRAG,2604.24618v1,Evaluating whether AI models would sabotage AI...,"model behavior,prefill awareness",Opus 4.7 Preview,False,False,,,
3,4,spaCy_REBEL,2604.24351v1,Diffusion Templates,use,Controllable diffusion,False,False,,,
4,5,spaCy_REBEL,2604.23530v1,MTRouter,instance of,LLM,False,False,,,


### Calcule de précision manuellement 

### 7.3 — Calcul de précision manuelle

Lecture du CSV `manual_validation_50_triplets.csv` (où l'utilisateur a rempli `manual_is_correct` avec 1 ou 0), puis calcul :
- **Précision globale** : % de triplets marqués corrects
- **Précision par système** : LightRAG vs REBEL séparément

Résultats sauvegardés en JSON `manual_validation_50_precision_report.json` pour documentation du rapport.

In [21]:
import json
import pandas as pd

MANUAL_CSV = OUT_DIR / "manual_validation_50_triplets.csv"
PRECISION_OUT = OUT_DIR / "manual_validation_50_precision_report.json"

df = pd.read_csv(MANUAL_CSV)

def parse_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s in {"1", "yes", "y", "oui", "o", "true", "correct"}:
        return 1
    if s in {"0", "no", "n", "non", "false", "incorrect"}:
        return 0
    return None

df["_label"] = df["manual_is_correct"].apply(parse_label)
validated = df[df["_label"].notna()].copy()

if len(validated) == 0:
    print("Aucun triplet validé. Remplis manual_is_correct dans:", MANUAL_CSV)
else:
    overall_precision = validated["_label"].mean()

    by_system = (
        validated
        .groupby("source_system")["_label"]
        .agg(["count", "sum", "mean"])
        .reset_index()
        .rename(columns={
            "count": "validated_triplets",
            "sum": "correct_triplets",
            "mean": "precision"
        })
    )

    report = {
        "validated_triplets": int(len(validated)),
        "correct_triplets": int(validated["_label"].sum()),
        "overall_precision": round(float(overall_precision), 4),
        "overall_precision_pct": round(float(overall_precision) * 100, 2),
        "precision_by_system": by_system.to_dict(orient="records")
    }

    PRECISION_OUT.write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print("=== Precision manuelle ===")
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print("Rapport sauvegardé:", PRECISION_OUT)

Aucun triplet validé. Remplis manual_is_correct dans: data\processed\triplets\manual_validation_50_triplets.csv


### 8.1 — Benchmark RAG avec HotpotQA et ChromaDB (Chroma Vector Store)

Évaluation complète d'un pipeline RAG naïf utilisant ChromaDB comme base vectorielle (alternative simple à LightRAG) :
1. **Load index** : Chroma stocké dans `indexes/chroma_pfe500_baseline/`
2. **Load HotpotQA** : 20 questions multi-sauts du benchmark
3. **Retrieve** : pour chaque question, récupérer top-5 chunks pertinents
4. **Generate** : appel Ollama pour générer réponse
5. **Evaluate** : exactitude (EM), recall support documents, RAGAS metrics (faithfulness, answer_relevancy, context_precision, context_recall)

Résultats exportés pour comparaison avec les autres méthodes.

In [26]:
import string
from datasets import load_dataset, Dataset

from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.run_config import RunConfig


CHROMA_DIR = r"indexes\chroma_pfe500_baseline"
COLLECTION_NAME = "pfe_500_baseline"

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "llama3.1:8b"
EMBED_MODEL = "nomic-embed-text"
OLLAMA_NUM_CTX = 8192

TOP_K = 5
N_SAMPLES = 20


emb_fn = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_URL)

vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=emb_fn,
    collection_name=COLLECTION_NAME,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})


hotpot = load_dataset("hotpot_qa", "distractor", split=f"validation[:{N_SAMPLES}]")


llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_URL,
    temperature=0,
    num_ctx=OLLAMA_NUM_CTX,
)


def normalize(x):
    x = str(x).lower()
    x = "".join(ch for ch in x if ch not in string.punctuation)
    return x


def rag(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)

    prompt = f"""
Answer ONLY using context.
If not found say "I don't know".

Context:
{context}

Question: {question}

Answer:
"""
    pred_msg = llm.invoke(prompt)
    pred = pred_msg.content if hasattr(pred_msg, "content") else str(pred_msg)
    return pred, docs


results = []

for i, ex in enumerate(hotpot):
    q = ex["question"]
    gold = ex["answer"]

    pred, docs = rag(q)

    retrieved_titles = [d.metadata.get("title") for d in docs]
    support = set(ex["supporting_facts"]["title"])
    recall = len(set(retrieved_titles) & support) / max(1, len(support))

    failure = (
        "retrieval_multi_hop_failure" if recall < 1 else
        "generation_failure" if normalize(gold) not in normalize(pred) else
        "success"
    )

    results.append({
        "question": q,
        "gold": gold,
        "pred": pred,
        "docs": docs,
        "recall": recall,
        "failure": failure
    })

    print(i, failure, recall)


ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for r in results:
    ragas_data["question"].append(r["question"])
    ragas_data["answer"].append(r["pred"])
    ragas_data["contexts"].append([d.page_content for d in r["docs"]])
    ragas_data["ground_truth"].append(r["gold"])

dataset = Dataset.from_dict(ragas_data)

run_config = RunConfig(timeout=900)

ragas_result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=llm,
    embeddings=emb_fn,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

print(ragas_result)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_39632\3888184278.py:8: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_39632\3888184278.py:8: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_39632\3888184278.py:8: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from

0 retrieval_multi_hop_failure 0.0
1 retrieval_multi_hop_failure 0.0
2 retrieval_multi_hop_failure 0.0
3 retrieval_multi_hop_failure 0.0
4 retrieval_multi_hop_failure 0.0
5 retrieval_multi_hop_failure 0.0
6 retrieval_multi_hop_failure 0.0
7 retrieval_multi_hop_failure 0.0
8 retrieval_multi_hop_failure 0.0
9 retrieval_multi_hop_failure 0.0
10 retrieval_multi_hop_failure 0.0
11 retrieval_multi_hop_failure 0.0
12 retrieval_multi_hop_failure 0.0
13 retrieval_multi_hop_failure 0.0
14 retrieval_multi_hop_failure 0.0
15 retrieval_multi_hop_failure 0.0
16 retrieval_multi_hop_failure 0.0
17 retrieval_multi_hop_failure 0.0
18 retrieval_multi_hop_failure 0.0
19 retrieval_multi_hop_failure 0.0


Evaluating:   1%|▏         | 1/80 [00:42<56:08, 42.64s/it]ERROR:ragas.executor:Exception raised in Job[1]: OutputParserException(Invalid json output: The answer 'I don't know.' is noncommittal, so I will generate a question and set noncommittal to 1.
Question: What do you not know?
Noncommittal: 1
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Evaluating:  11%|█▏        | 9/80 [06:12<49:34, 41.89s/it]  ERROR:ragas.executor:Exception raised in Job[9]: OutputParserException(Failed to parse StringIO from completion null. Got: 1 validation error for StringIO
  Input should be a valid dictionary or instance of StringIO [type=model_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/model_type
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Evaluating:  26%|██▋       | 21/80 [19:51<38:06, 38.75s/it]   ERROR:raga

{'faithfulness': 0.1389, 'answer_relevancy': 0.0000, 'context_precision': 0.0000, 'context_recall': 0.0250}


### 8.2 — Mise à jour des dépendances critiques

Réinstallation forcée des packages essentiels (langchain-community, ragas) avec versions précises pour résoudre les conflits d'importation détectés lors de l'exécution des cellules d'évaluation.

In [2]:
import sys
!{sys.executable} -m pip install -U pip setuptools wheel
!{sys.executable} -m pip install --no-deps --ignore-installed --no-cache-dir "langchain-community==0.4.1"
!{sys.executable} -m pip install -U --no-cache-dir ragas

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.5 MB 6.7 MB/s eta 0:00:01
   -------------------- ------------------- 1.3/2.5 MB 3.5 MB/s eta 0:00:01
   ---------------------------- ----------- 1.8/2.5 MB 3.4 MB/s eta 0:00:01
   ------------------------------------- -- 2.4/2.5 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 3.0 MB/s  0:00:00
